In [1]:
import datetime
import requests
from ddgs import DDGS
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import AIMessageChunk
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
llm = ChatOllama(model="llama3.2:3b", temperature=0)

config = {"configurable": {"thread_id": "1"}}

In [3]:
@tool
def calculator(expression: str) -> str:
    """This is a calculator tool that evaluates mathematical expressions.
    Args:
        expression (str): The mathematical expression to evaluate.
    Returns:
        str: The result of the calculation.
    e.g. "2 + 2" -> "4"
    """
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "Invalid Expression"
    return str(eval(expression))

@tool
def get_current_time() -> str:
    """This is a tool that returns the current time.
    Returns:
        str: The current time in HH:MM:SS format.
    e.g. "14:30:15"
    """
    return datetime.datetime.now().strftime("%H:%M:%S")


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city. Use it when the user asks about
    weather or temperature. Input is a city name in English, e.g. 'Cairo'."""
    try:
        # 1) اسم المدينة -> إحداثيات
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=10,
        ).json()
        if not geo.get("results"):
            return f"Could not find city: {city}"
        place = geo["results"][0]

        w = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
            },
            timeout=10,
        ).json()["current"]

        return (
            f"{place['name']}, {place.get('country', '')}: "
            f"{w['temperature_2m']}°C, humidity {w['relative_humidity_2m']}%, "
            f"wind {w['wind_speed_10m']} km/h"
        )
    except Exception as e:
        return f"Weather lookup failed: {e}"

@tool
def web_search(query: str) -> str:
    """Search the internet for current information, news, or recent facts.
    Input is a short, specific English query, e.g. 'OpenAI new model release'."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.news(query, max_results=5))
        if not results:
            return "No results found."
        return "\n\n".join(
            f"Title: {r['title']}\nDate: {r.get('date', '')}\n"
            f"URL: {r['url']}\nSnippet: {r['body']}"
            for r in results
        )
    except Exception as e:
        return f"Search failed: {e}"

tools = [calculator, get_current_time, get_weather, web_search]

In [4]:
agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt = """You are a helpful assistant with access to these tools:

- calculator: use it for ANY arithmetic. Never calculate in your head.
- get_current_time: use it when asked about the current time.
- get_weather: use it when asked about weather or temperature. Pass the city name in English (e.g. "Cairo").
- web_search: use it for news, recent events, or facts that may have changed. Pass a short English query.

Rules:
1. Call a tool only when it is needed. For general knowledge or casual chat, answer directly.
2. Never invent tool results. Use only what the tool returns.
3. After web_search, base your answer on the results and mention the source URL.
4. If a tool fails or returns nothing, say so honestly.
5. Always reply in the same language the user writes in (Arabic or English).
6. Keep answers short and clear."""
    ,
    checkpointer=InMemorySaver()
)


In [5]:
def stream_answer(question, config):
    for chunk, meta in agent.stream(
        {'messages': [('user', question)]},
        config,
        stream_mode = 'messages',
    ):
        if isinstance(chunk, AIMessageChunk) and chunk.content:
            print(chunk.content, end="", flush=True)
    print()

In [6]:

if __name__ == "__main__":
    stream_answer("The weather in Tanta?", config)
    stream_answer("What is 123 * 456?", config)
    stream_answer("What is the current time?", config)
    stream_answer("Search for the latest news about AI.", config)

The current weather in Tanta, Egypt is 25.8°C with a relative humidity of 41% and a moderate wind speed of 13.2 km/h.
The result of 123 multiplied by 456 is 56088.
The current time is 4:02 PM.
The latest news about AI includes:

* AI, tech, and data centers are shaping key 2026 midterm races in Ohio, Michigan, Pennsylvania, and Wisconsin (MSN, 10/18/2026)
* AI data-center stocks slide, renewables rally after Democratic US midterm sweep (MSN, 10/15/2026)
* World shares mostly gain after US midterm elections as AI-related stocks sink (MSN, 11/01/2026)
* Scammers are using AI to fake images of lost pets, demand ransom for their return, according to report (Mashable, 11/04/2026)
* Democratic sweep adds new twist to markets' Iran, AI, and debt worries (Reuters, 10/16/2026)

Source URLs:

* https://www.msn.com/en-us/news/other/these-races-matter-if-you-care-about-money-musk-and-the-ai-boom-as-democrats-rack-up-shocking-wins/ar-AA2gGntT?ocid=BingNewsVerp
* https://www.msn.com/en-us/money/top-